# Jaanch Job-Scam Detector: Model v3 Tuned (Hyperparameter Optimization)

**Version**: `v3-tuned` (TF-IDF FeatureUnion + LogisticRegression + Sigmoid Calibration)  
**Notebook**: `16_train_model_v3_tuned.ipynb`  
**Goal**: Perform extensive hyperparameter tuning over the word-level and character-level `FeatureUnion` space using only the existing training pool `jaanch_training_pool_v3.csv`.

### Strict Operational Invariants (Never Violated)
1. **No Data Generation or Scrapes**: Uses only the existing frozen training pool `jaanch_training_pool_v3.csv`.
2. **Zero-Leakage Invariant**: The local held-out test set (`jaanch_local_test_set.csv`, N=120) is strictly isolated and evaluated only once at the final comparison step.
3. **Safety Guardrails**: ML signal is strictly capped at `medium` severity. ML can never trigger High Risk, mark a posting Verified, or override rule findings.
4. **Custom Jaanch Scorer**: Optimizes for high scam recall and low false-positive rate on genuine postings, not plain accuracy.


In [ ]:
import os
import sys
import json
import time
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, brier_score_loss, roc_curve, precision_recall_curve, make_scorer
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Kaggle / Local Output Directory Detection
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
(OUTPUT_DIR / "reports").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)

print(f"Working output directory: {OUTPUT_DIR.resolve()}")


In [ ]:
def find_file(filename: str) -> Path:
    candidates = [
        Path(f"/kaggle/input/{filename}"),
        Path(f"/kaggle/input/jaanch-assets/{filename}"),
        Path(f"/kaggle/input/jaanch/{filename}"),
        Path(f"/kaggle/working/{filename}"),
        Path(f"data/{filename}"),
        Path(f"ml/data/{filename}"),
        Path(f"artifacts/models/{filename}"),
        Path(f"models/{filename}"),
        Path(f"reports/{filename}"),
        Path(f"ml/reports/{filename}"),
        Path(filename)
    ]
    for c in candidates:
        if c.exists():
            return c
    # Recursive search under /kaggle/input and current workspace
    for root in [Path("/kaggle/input"), Path(".")]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Could not locate required asset: {filename}")

pool_path = find_file("jaanch_training_pool_v3.csv")
test_path = find_file("jaanch_local_test_set.csv")
print(f"Located training pool: {pool_path}")
print(f"Located local test set: {test_path}")


## Step 1: Load Existing Data & Audit Distributions
- Load `jaanch_training_pool_v3.csv`.
- Validate required schema (`text`, `label`, `source`, `category`).
- Remove empty text, text under 50 characters, exact duplicates, and invalid labels.
- Verify strict zero-leakage exclusion against `jaanch_local_test_set.csv`.


In [ ]:
df_raw = pd.read_csv(pool_path)
df_test = pd.read_csv(test_path)

print(f"Initial raw training pool shape: {df_raw.shape}")
print(f"Held-out local test set shape: {df_test.shape}")

# Schema validation
for col in ["text", "label", "source", "category"]:
    assert col in df_raw.columns, f"Missing required column: {col}"

# Cleaning: trim, length >= 50, valid labels
df_clean = df_raw.copy()
df_clean["text"] = df_clean["text"].astype(str).str.strip()
df_clean = df_clean[df_clean["text"].str.len() >= 50]
df_clean = df_clean[df_clean["label"].isin(["scam", "genuine"])]
df_clean = df_clean.drop_duplicates(subset=["text"])

# Zero-leakage guarantee
test_texts = set(df_test["text"].astype(str).str.strip())
pre_check = len(df_clean)
df_clean = df_clean[~df_clean["text"].isin(test_texts)]
print(f"Test set overlap: {pre_check - len(df_clean)} rows excluded (Strict Zero-Leakage).")

df_clean["target"] = (df_clean["label"] == "scam").astype(int)

print(f"Cleaned pool size: {len(df_clean)}")
print("\nLabel distribution:\n", df_clean["label"].value_counts())
print("\nSource distribution:\n", df_clean["source"].value_counts())
print("\nTop categories:\n", df_clean["category"].value_counts().head(10))

lengths = df_clean["text"].str.len()
print(f"\nText length: min={lengths.min()}, 25%={lengths.quantile(0.25):.0f}, median={lengths.median():.0f}, 75%={lengths.quantile(0.75):.0f}, max={lengths.max()}")

# Stratified Train/Validation split (85/15)
train_df, val_df = train_test_split(
    df_clean,
    test_size=0.15,
    stratify=df_clean["target"],
    random_state=RANDOM_STATE
)
print(f"\nTrain split: {len(train_df)} | Val split: {len(val_df)}")


## Step 2: Build Tunable Pipeline
Combine word-level TF-IDF and character boundary TF-IDF (`char_wb`) using `FeatureUnion`, followed by `LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)`.


In [ ]:
base_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(stop_words="english")),
        ("char", TfidfVectorizer(analyzer="char_wb"))
    ])),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

print("Pipeline initialized:")
print(base_pipeline)


## Step 3: Hyperparameter Search
Define a custom optimization scorer optimizing Jaanch's core priorities:
1. High scam recall (detecting fraud).
2. Low false positive rate on genuine postings (protecting genuine applicants).
3. Balanced F1-score.


In [ ]:
def custom_jaanch_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    
    # Composite objective: F1 + Recall - 2.0 * FPR penalty
    return float((0.45 * f1) + (0.45 * recall) - (2.0 * fpr))

scorer = make_scorer(custom_jaanch_scorer, greater_is_better=True)

param_grid = {
    "features__word__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "features__word__min_df": [1, 2, 3],
    "features__word__max_features": [40000, 60000, 80000],
    "features__word__sublinear_tf": [True, False],
    "features__char__ngram_range": [(2, 5), (3, 5), (3, 6)],
    "features__char__min_df": [1, 2, 3],
    "features__char__max_features": [40000, 60000, 80000],
    "features__char__sublinear_tf": [True, False],
    "clf__C": [0.1, 0.25, 0.5, 0.8, 1.0, 2.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Starting RandomizedSearchCV (40 iterations, 5-fold Stratified CV)...")
search = RandomizedSearchCV(
    estimator=base_pipeline,
    param_distributions=param_grid,
    n_iter=40,
    scoring=scorer,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=2
)

t0 = time.time()
search.fit(train_df["text"], train_df["target"])
print(f"Tuning finished in {time.time() - t0:.1f}s")
print(f"Best CV Jaanch Scorer: {search.best_score_:.4f}")
print("Best Hyperparameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")


## Step 4: Calibrate the Best Model
Fit best pipeline on entire training partition, then calibrate using `CalibratedClassifierCV(method='sigmoid', cv=5)`.


In [ ]:
best_pipeline = search.best_estimator_
best_pipeline.fit(train_df["text"], train_df["target"])

calibrated_v3_tuned = CalibratedClassifierCV(
    estimator=best_pipeline,
    method="sigmoid",
    cv=5
)
calibrated_v3_tuned.fit(train_df["text"], train_df["target"])

# Save model artifacts
uncal_path = OUTPUT_DIR / "jaanch_scam_model_v3_tuned.joblib"
cal_path = OUTPUT_DIR / "jaanch_scam_model_v3_tuned_calibrated.joblib"
hp_path = OUTPUT_DIR / "v3_tuned_best_hyperparameters.json"

joblib.dump(best_pipeline, uncal_path)
joblib.dump(calibrated_v3_tuned, cal_path)

serializable_params = {k: list(v) if isinstance(v, tuple) else v for k, v in search.best_params_.items()}
with open(hp_path, "w") as f:
    json.dump({
        "best_cv_score": float(search.best_score_),
        "best_params": serializable_params,
        "selected_at": datetime.now().isoformat()
    }, f, indent=2)

print(f"Saved: {uncal_path}")
print(f"Saved: {cal_path}")
print(f"Saved: {hp_path}")


## Step 5: Tune Decision Threshold
Using only the validation split, evaluate candidate decision thresholds from 0.30 to 0.90 in increments of 0.05.


In [ ]:
val_probs = calibrated_v3_tuned.predict_proba(val_df["text"])[:, 1]

records = []
for thresh in np.arange(0.30, 0.95, 0.05):
    t = round(float(thresh), 2)
    preds = (val_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_df["target"], preds, labels=[0, 1]).ravel()
    prec = precision_score(val_df["target"], preds, zero_division=0)
    rec = recall_score(val_df["target"], preds, zero_division=0)
    f1 = f1_score(val_df["target"], preds, zero_division=0)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    
    records.append({
        "threshold": t,
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "f1_score": round(float(f1), 4),
        "fpr_on_genuine": round(float(fpr), 5),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "medium_warnings": int(np.sum(val_probs >= t))
    })

thresh_df = pd.DataFrame(records)
print(thresh_df.to_string(index=False))

thresh_csv_path = OUTPUT_DIR / "v3_tuned_threshold_analysis.csv"
thresh_df.to_csv(thresh_csv_path, index=False)

# Establish Policy
policy_dict = {
    "policy_name": "jaanch_conservative_triage_v3_tuned",
    "low_threshold": 0.50,
    "medium_threshold": 0.75,
    "max_allowed_severity": "medium",
    "policy_invariants": {
        "can_trigger_high": False,
        "can_mark_verified": False,
        "can_override_rule": False
    }
}
with open(OUTPUT_DIR / "v3_tuned_threshold_policy.json", "w") as f:
    json.dump(policy_dict, f, indent=2)


## Step 6: Final Comparison on Untouched Local Test Set (N=120)
Evaluate both the current v3 calibrated model and the new v3-tuned calibrated model on the untouched test set.


In [ ]:
v3_current_path = find_file("jaanch_scam_model_v3_calibrated.joblib")
model_v3_current = joblib.load(v3_current_path)

X_test = df_test["text"].astype(str).str.strip()
y_test = (df_test["label"] == "scam").astype(int)

probs_curr = model_v3_current.predict_proba(X_test)[:, 1]
probs_tuned = calibrated_v3_tuned.predict_proba(X_test)[:, 1]

preds_curr = (probs_curr >= 0.50).astype(int)
preds_tuned = (probs_tuned >= 0.50).astype(int)

def evaluate_metrics(y_true, probs, preds):
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    return {
        "accuracy": round(float(accuracy_score(y_true, preds)), 4),
        "precision": round(float(precision_score(y_true, preds, zero_division=0)), 4),
        "recall": round(float(recall_score(y_true, preds, zero_division=0)), 4),
        "f1_score": round(float(f1_score(y_true, preds, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(y_true, probs)), 4),
        "brier_score": round(float(brier_score_loss(y_true, probs)), 4),
        "fpr": round(float(fp / (fp + tn)), 4),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)
    }

m_c = evaluate_metrics(y_test, probs_curr, preds_curr)
m_t = evaluate_metrics(y_test, probs_tuned, preds_tuned)

comp_df = pd.DataFrame([m_c, m_t], index=["v3-Current", "v3-Tuned"]).T
comp_df["Delta (Tuned - Current)"] = comp_df["v3-Tuned"] - comp_df["v3-Current"]
print("Overall Metrics Comparison on Held-Out Test Set (N=120):")
print(comp_df)


In [ ]:
# Category Level Breakdown
cat_records = []
for cat, grp in df_test.groupby("category"):
    idx = grp.index
    p_c = probs_curr[idx]
    p_t = probs_tuned[idx]
    is_scam = (grp["label"].iloc[0] == "scam")
    cat_records.append({
        "category": cat,
        "type": "scam" if is_scam else "genuine",
        "n": len(grp),
        "current_mean_prob": round(float(np.mean(p_c)), 4),
        "tuned_mean_prob": round(float(np.mean(p_t)), 4),
        "delta": round(float(np.mean(p_t) - np.mean(p_c)), 4)
    })

cat_summary_df = pd.DataFrame(cat_records)
print("\nCategory Separation Summary:")
print(cat_summary_df.to_string(index=False))

# Plot ROC and PR Curves
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
fpr_c, tpr_c, _ = roc_curve(y_test, probs_curr)
fpr_t, tpr_t, _ = roc_curve(y_test, probs_tuned)
plt.plot(fpr_c, tpr_c, label=f"v3-current (AUC={m_c['roc_auc']:.4f})", color="steelblue", lw=2)
plt.plot(fpr_t, tpr_t, label=f"v3-tuned (AUC={m_t['roc_auc']:.4f})", color="forestgreen", lw=2, linestyle="--")
plt.plot([0, 1], [0, 1], "k:", alpha=0.5)
plt.title("ROC Curve Comparison", fontsize=11, fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
prec_c, rec_c, _ = precision_recall_curve(y_test, probs_curr)
prec_t, rec_t, _ = precision_recall_curve(y_test, probs_tuned)
plt.plot(rec_c, prec_c, label=f"v3-current (F1={m_c['f1_score']:.4f})", color="steelblue", lw=2)
plt.plot(rec_t, prec_t, label=f"v3-tuned (F1={m_t['f1_score']:.4f})", color="forestgreen", lw=2, linestyle="--")
plt.title("Precision-Recall Curve Comparison", fontsize=11, fontweight="bold")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
chart_path = OUTPUT_DIR / "reports" / "v3_vs_v3_tuned_curves.png"
plt.savefig(chart_path, dpi=300)
plt.show()


## Step 7: Deployment Recommendation
Evaluate conditions:
- Fewer or equal false positives on genuine postings
- Improved scam recall or category-level separation
- No material regression in F1-score
- Acceptable probability calibration


In [ ]:
cond_fp = (m_t["fp"] <= m_c["fp"])
cond_rec = (m_t["recall"] >= m_c["recall"])
cond_f1 = (m_t["f1_score"] >= m_c["f1_score"] - 0.005)
cond_cal = (m_t["brier_score"] <= 0.030)

print(f"Condition 1 (FP <= Baseline): {cond_fp}")
print(f"Condition 2 (Scam Recall Retained): {cond_rec}")
print(f"Condition 3 (F1 Score Stability): {cond_f1}")
print(f"Condition 4 (Calibration Score <= 0.030): {cond_cal}")

if cond_fp and cond_rec and cond_f1 and cond_cal:
    recommendation = "deploy_v3_tuned_with_monitoring"
elif cond_fp and cond_f1:
    recommendation = "keep_current_v3"
else:
    recommendation = "do_not_deploy"

print(f"\nFINAL RECOMMENDATION: {recommendation.upper()}")
